# IEEE-CIS Fraud Detection — Exploratory Data Analysis

## Goal
Understand the structure and characteristics of the IEEE-CIS fraud dataset
to inform all subsequent modeling and architecture decisions.

## Dataset
- Source: IEEE-CIS Fraud Detection — Vesta Corporation / Kaggle Competition
- Files: train_transaction.csv + train_identity.csv
- Size: ~590,540 transactions, 394 features across both files

## Key Questions
1. How imbalanced is the fraud label? — defines training strategy
2. What is the structure of transaction vs. identity data? — defines join strategy
3. Which feature groups (C, D, M, V) have the most missing values? — defines preprocessing
4. What is the distribution of TransactionAmt? — defines scaling strategy
5. What temporal patterns exist in TransactionDT? — defines feature engineering

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

import stats

sns.set_theme(style="whitegrid")

print("Libraries loaded successfully")

Libraries loaded successfully


In [80]:
# Load file with time

def loadFile(file_name):
    start = time.time()
    df = pd.read_csv(file_name, low_memory=False)
    load_time = time.time() - start
    return df, load_time

transaction_file_name= "../data/train_transaction.csv"
dftx, load_time_tx = loadFile(transaction_file_name)

identity_file_name= "../data/train_identity.csv"
dfid, load_time_id = loadFile(identity_file_name)


def dataset_summary(df, name, load_time):
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"=== {name} ===")
    print(f"Shape:      {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"Memory:     {mem_mb:.1f} MB")
    print(f"Load time:  {load_time:.2f}s")
    print()

dataset_summary(dftx, "train_transaction", load_time_tx)
dataset_summary(dfid, "train_identity", load_time_id)

=== train_transaction ===
Shape:      590,540 rows x 394 cols
Memory:     2062.1 MB
Load time:  18.87s

=== train_identity ===
Shape:      144,233 rows x 41 cols
Memory:     143.1 MB
Load time:  0.43s



In [81]:
print(dftx.head())

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  0.0   0.0   0.0   0.0   0.0  0.0   

  V336  V337  V338  V339  
0  NaN   NaN   NaN   NaN  
1  NaN   NaN   NaN  

In [82]:
dftx['isFraud'].value_counts(dropna=False)

isFraud
0    569877
1     20663
Name: count, dtype: int64

### Outcome feature and baseline

- **'isFraud'** is the outcome feature with value 0 if the transaction is not fraudulent and 1 if it is. There is no NaN values. The **baseline** is only 3.4% of the total amount of transactions are fraudulent.

### Feature Groups and Interpretation
- **`TransactionDT`** — not a standard Unix timestamp. It's a timedelta in seconds from an unknown reference point. This matters: it can't be converted directly to a real date, but time differences and cyclical patterns (hour of day, day of week) can be derived if the right granularity is assumed.

- **`TransactionAmt`** — transaction amount. One of the few fully interpretable features. Heavily skewed distribution — expect extreme outliers.

- **`ProductCD`** — product category (W, H, C, S, R). Obfuscated, but appears to behave in an ordinal-ish way with respect to fraud according to the Kaggle community.

- **`card1`–`card6`** — card information. A combination of card type, issuing bank, country, etc. `card1` is likely a hashed card number — high cardinality, useful for frequency-based feature engineering.

- **`addr1`, `addr2`** — buyer address and billing country. `addr1` has ~300 unique values (US zip code), `addr2` ~100 (country).

- **`dist1`, `dist2`** — distances. Likely between billing and shipping address, or between previous transactions from the same customer. High null rate.

- **`P_emaildomain`, `R_emaildomain`** — email domain of the purchaser (P) and recipient (R). Useful: gmail vs. yahoo vs. corporate domains have different fraud profiles.

- **`C1`–`C14`** — counting features. How many times a card, address, etc. has been seen. The exact definitions aren't documented, but they're aggregations of historical behavior. Highly predictive.

- **`D1`–`D15`** — time deltas. Days since the previous transaction across different dimensions (same card, same device, etc.). Many nulls = first time that entity is seen.

- **`M1`–`M9`** — binary match features (T/F/NaN). Does the name on the card match the order name? Does the address match? Nulls here carry their own meaning — not "missing data", but "couldn't be verified".

- **`V1`–`V339`** — the most opaque group. Generated internally by Vesta. Likely device behavior, velocity, and merchant historical pattern features. Many are correlated with each other and follow block-wise null patterns (if `V1` is null, `V2`–`V11` are probably null too — they come in groups).

- **Identity features (`id_01`–`id_38`, `DeviceType`, `DeviceInfo`)** — from the identity file. Only ~24% of transactions have identity data. That's not noise — the absence of identity data is itself a pattern.

In [83]:
dftx.dtypes.value_counts()

float64    376
str         14
int64        4
Name: count, dtype: int64

In [84]:
dftx.select_dtypes(include='object').columns

/var/folders/wn/nj1blbjd4g5b3cl1hg7vr2bh0000gn/T/ipykernel_77080/4069716484.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  dftx.select_dtypes(include='object').columns


Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1',
       'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'],
      dtype='str')


### Categorical features:
- We can see only 14 features are not numeric. Let us going to analyse one by one.

In [85]:
dftx[['M1','M2','M3','M5', 'M6','M7','M8','M9']].head()

,M1,M2,M3,M5,M6,M7,M8,M9
0,T,T,T,F,T,NaN,NaN,NaN
1,NaN,NaN,NaN,T,T,NaN,NaN,NaN
2,T,T,T,F,F,F,F,F
3,NaN,NaN,NaN,T,F,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
print(dftx['M4'].value_counts(dropna=False))
temp = dftx['M4'].fillna('NaN_value')
pd.crosstab(temp, dftx['isFraud'], normalize='index').sort_values(by=1)

M4
NaN    281444
M0     196405
M2      59865
M1      52826
Name: count, dtype: int64


isFraud,0,1
M4,,
NaN_value,0.981428,0.018572
M1,0.972949,0.027051
M0,0.963351,0.036649
M2,0.886261,0.113739


### Exploring features: M1-M9

- **M4** - It is a column with 4 different values. We don't really know what the feature means but we can see value 'M2' is categorized as fraud in 11%. It is an interesting value compared with the baseline (3.4%). It has a different context from M1-M9 to check. We are going to leave it to do a post-training study about how important it is.
- **M1-M9** are binary match features (T/F/NaN) but the reality is that they are ternary features. One simple change is to transform into two boolean columns (M1_Known : T-has value, F - NaN and M1_Value: T/F only make sense if M1_known is T)

In [87]:
print(f"=== Card 4 ===")
print(f"Categories and counts",dftx['card4'].value_counts(dropna=False))

print(f"=== Card 6 ===")
print(f"Categories and counts",dftx['card6'].value_counts(dropna=False))


=== Card 4 ===
Categories and counts card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64
=== Card 6 ===
Categories and counts card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64


### Exploring features: Card 4 and Card 6

- **card 4**: Represent the **Card Vendor** with 0.27% of null values. Initial decision: 5 columns one-hot encoding (visa, master, american, discover, unknown). Take into account apply the drop-first.
- **card 6**: Identifies the **card type** (debit, credit, debit or credit, charge card). We can see there are so few values for `debit or credit` (30)  and `charge card` (15). We have two options here, delete those values because represent a 0.007% of the dataset or group `debit or credit` and `charge card` in a new category called `other`, I mean, it is not `debit` nor `credit`. We finally decided not to remove the values to not lose the information and avoid the **unseen categories** problem. There is a 0.27% of null values. Initial decision: 3 columns one-hot encoding (debit, credit, other). Due to the small percentage of nulls, we can set the `unknown` values into the `other` category. Take into account apply the drop-first.

In [88]:
print(f"=== P_emaildomain ===")
print(f"Categories and counts",dftx['P_emaildomain'].value_counts(dropna=False))

print(f"=== R_emaildomain ===")
print(f"Categories and counts",dftx['R_emaildomain'].value_counts(dropna=False))

=== P_emaildomain ===
Categories and counts P_emaildomain
gmail.com           228355
yahoo.com           100934
NaN                  94456
hotmail.com          45250
anonymous.com        36998
aol.com              28289
comcast.net           7888
icloud.com            6267
outlook.com           5096
msn.com               4092
att.net               4033
live.com              3041
sbcglobal.net         2970
verizon.net           2705
ymail.com             2396
bellsouth.net         1909
yahoo.com.mx          1543
me.com                1522
cox.net               1393
optonline.net         1011
charter.net            816
live.com.mx            749
rocketmail.com         664
mail.com               559
earthlink.net          514
gmail                  496
outlook.es             438
mac.com                436
juno.com               322
aim.com                315
hotmail.es             305
roadrunner.com         305
windstream.net         305
hotmail.fr             295
frontier.com           2

In [89]:
temp = dftx['P_emaildomain'].fillna('NaN_value')
pd.crosstab(temp, dftx['isFraud'], normalize='index').sort_values(by=1)

isFraud,0,1
P_emaildomain,,
hotmail.de,1.000000,0.000000
hotmail.co.uk,1.000000,0.000000
yahoo.co.jp,1.000000,0.000000
windstream.net,1.000000,0.000000
hotmail.fr,1.000000,0.000000
web.de,1.000000,0.000000
yahoo.de,1.000000,0.000000
gmx.de,1.000000,0.000000
cfl.rr.com,1.000000,0.000000


In [90]:
temp = dftx['R_emaildomain'].fillna('NaN_value')
pd.crosstab(temp, dftx['isFraud'], normalize='index')

isFraud,0,1
R_emaildomain,,
NaN_value,0.979181,0.020819
aim.com,0.972222,0.027778
anonymous.com,0.970870,0.029130
aol.com,0.965145,0.034855
att.net,1.000000,0.000000
...,...,...
yahoo.com.mx,0.989390,0.010610
yahoo.de,1.000000,0.000000
yahoo.es,0.964912,0.035088


In [91]:
dftx['R_emaildomain'].value_counts()

R_emaildomain
gmail.com           57147
hotmail.com         27509
anonymous.com       20529
yahoo.com           11842
aol.com              3701
outlook.com          2507
comcast.net          1812
yahoo.com.mx         1508
icloud.com           1398
msn.com               852
live.com              762
live.com.mx           754
verizon.net           620
me.com                556
sbcglobal.net         552
cox.net               459
outlook.es            433
att.net               430
bellsouth.net         422
hotmail.fr            293
hotmail.es            292
web.de                237
mac.com               218
prodigy.net.mx        207
ymail.com             207
optonline.net         187
gmx.de                147
yahoo.fr              137
charter.net           127
mail.com              122
hotmail.co.uk         105
gmail                  95
earthlink.net          79
yahoo.de               75
rocketmail.com         69
embarqmail.com         68
scranton.edu           63
yahoo.es               5

### Exploring features: R_emaildomain and P_emaildomain

- **P_emaildomain**
    - 60 unique domains + NaN
    - 94,456 null values (16% of the dataset)
    - 36,998 records with anonymous.com (6.3%)
    - Fraud rate by relevant categories:
        - NaN: 2.1%
        - anonymous.com: 2.9%
        - protonmail.com: 40% — only 76 records (0.013% of total, statistically irrelevant)
        - Remaining domains: between 1.3% and 3.8% — no clear separation from the 3.4% baseline

- **R_emaildomain**
    - High proportion of nulls: 453,249 (76.7% of the dataset) — most transactions have no identified recipient email
    - anonymous.com: 20,529 records, fraud rate 2.9%
    - Individual domains show no consistent fraud signal

- Conclusion
Neither feature shows a clear and consistent correlation with isFraud. The only exception is protonmail.com in P_emaildomain with a 40% fraud rate, but with only 76 records it is statistically irrelevant. **Decision: drop both features.**

In [92]:
dftx['ProductCD'].value_counts(dropna=False)

ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

In [93]:
pd.crosstab(dftx['ProductCD'], dftx['isFraud'], normalize='index')

isFraud,0,1
ProductCD,,
C,0.883127,0.116873
H,0.952338,0.047662
R,0.962174,0.037826
S,0.941004,0.058996
W,0.979601,0.020399


### Exploring features: ProductCD

- We have 5 different values (C,H,R,S,W) without NaN
- W has the highest count - 74.45% of the dataset
- C has an 11.68% of fraud values (3x the baseline)
- The other values seems not to be very decision-makers

- **Conclusion**: Transform the categorical feature to 5 one-hot features. Not finally needed if we are gonna use XGBoost because it natively manages the categorical features